# Baseline Train

**Descripción**: Comienzo con un experimento base para fijar un baseline con el entrenamiento de un algoritmo de regresión logística.  

Se setea la semilla 42 para poder reproducir resultados.

Se loguean los resultados de los experimentos con MLflow.

---

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from sklearn.metrics import (classification_report, roc_auc_score,
                              accuracy_score, precision_score,
                              recall_score, f1_score)
import mlflow
import mlflow.sklearn

# --- Carga de datos ---

X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')
y_test  = pd.read_parquet('../data/processed/y_test.parquet')



### 1. Imputación de datos faltantes y escalamiento
El algoritmo de regresión logística no admite nulos por lo que es necesario imputarlos, no entiende variables categóricas y es sensible a la escala. Se define un pipeline de sklearn que aplica transformaciones diferenciadas según el tipo de cada variable.

#### Clasificación de variables

| Grupo | Variables | Criterio |
|---|---|---|
| **Numéricas** | `price`, `initial_quantity`, `sold_quantity`, `pictures_count`, `pictures_max_area`, `non_mercado_pago_payment_methods_count`, `tags_count`, `attributes_count`, `variations_count` | Variables continuas o de conteo que requieren escalado |
| **Categóricas baja cardinalidad** | `warranty`, `listing_type_id`, `shipping_mode`, `buying_mode`, `state` | Pocas categorías, Ordinal Encoding es suficiente |
| **Categóricas alta cardinalidad** | `category_id`, `city` | Muchas categorías únicas — One Hot Encoding generaría demasiadas dimensiones |
| **Binarias** | `automatic_relist`, `shipping_free`, `shipping_local_pick_up`, `deal_ids`, `official_store_id`, `video_id` | Ya están en formato 0/1, solo requieren imputación |

#### Decisiones de transformación

**Variables numéricas — `SimpleImputer(median)` + `StandardScaler`**
- La mediana es más robusta que la media ante outliers, que son frecuentes en `price` e `initial_quantity`
- `StandardScaler` es necesario para la regresión logística ya que el optimizador es sensible a la escala — variables con rangos muy distintos sin escalar sesgan los coeficientes

**Categóricas baja cardinalidad — `SimpleImputer(most_frequent)` + `OrdinalEncoder`**
- Se imputa con la moda para no perder filas
- `OrdinalEncoder` asigna un entero a cada categoría sin expandir dimensiones, no siempre es la mejor opción ya que le da al modelo un orden que realmente no existe, pero evita usar one hot que en este caso generaría muchas features.
- `handle_unknown='use_encoded_value', unknown_value=-1` — si en test aparece una categoría no vista en train, se asigna -1 en lugar de generar error

**Binarias — `SimpleImputer(most_frequent)`**
- Ya están codificadas como 0/1, no requieren encoding
- Se imputa con la moda (el valor más frecuente entre 0 y 1)

**Categóricas alta cardinalidad — `SimpleImputer(constant='unknown')` + `TargetEncoder(smoothing=10)`**
- One Hot Encoding generaría cientos de columnas para `category_id` y miles para `city`
- Target Encoding reemplaza cada categoría por la probabilidad media del target, capturando la señal en una sola dimensión
- `smoothing=10` mezcla la media de la categoría con la media global según el número de ejemplos — reduce el overfitting en categorías con pocos registros (especialmente importante para `city` con 3656 valores únicos)
- Se imputa con `'unknown'` antes del encoding para que las categorías nulas reciban la media global como valor

#### Pipeline integrado con `ColumnTransformer`

Todas las transformaciones se encapsulan en un `ColumnTransformer` y luego en un `Pipeline`
junto con el clasificador. Esto garantiza que:

1. El `fit` de cada transformer ocurre **solo sobre train** — sin leakage
2. El `transform` de test usa los parámetros aprendidos en train (media, moda, mappings)
3. El pipeline completo es serializable y reproducible



In [2]:
# --- Definir columnas por tipo ---
cat_high_card = ['category_id', 'city']
cat_low_card  = ['warranty', 'listing_type_id', 'shipping_mode', 'buying_mode', 'state']
binary_cols   = ['automatic_relist', 'shipping_free',
                 'shipping_local_pick_up', 'deal_ids', 'official_store_id', 'video_id']
num_cols      = ['price', 'initial_quantity', 'sold_quantity',
                 'pictures_count', 'pictures_max_area',
                 'non_mercado_pago_payment_methods_count',
                 'tags_count', 'attributes_count', 'variations_count']

# --- Transformers ---
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_low_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# TargetEncoder se aplica por separado porque necesita y_train
high_card_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', TargetEncoder(smoothing=10))
])

# --- ColumnTransformer ---
preprocessor = ColumnTransformer(transformers=[
    ('num',      num_transformer,      num_cols),
    ('cat_low',  cat_low_transformer,  cat_low_card),
    ('binary',   binary_transformer,   binary_cols),
    ('cat_high', high_card_transformer, cat_high_card),
])

### 2. Entrenamiento, evaluación y logging con MLflow

Como punto de partida se entrena un modelo de regresión logística, modelo lineal simple que permite
establecer un baseline antes de pasar a modelos más complejos.

A su vez se utiliza MLflow para registrar el experimento, lo que incluye parámetros, métricas y el modelo entrenado. Esto facilita la comparación entre diferentes experimentos.

In [3]:

# --- Pipeline completo ---
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# --- Encodear target ---
y_train_enc = (y_train.iloc[:, 0] == 'new').astype(int)
y_test_enc  = (y_test.iloc[:, 0]  == 'new').astype(int)

# --- MLflow ---
mlflow.set_tracking_uri('../models/mlruns')
mlflow.set_experiment('Baseline Model')

with mlflow.start_run(run_name='logistic_regression_baseline'):

    mlflow.log_params({
        'model':                     'LogisticRegression',
        'max_iter':                  1000,
        'random_state':              42,
        'target_encoder_smoothing':  10,
        'imputer_num':               'median',
        'imputer_cat':               'most_frequent',
    })

    pipeline.fit(X_train, y_train_enc)
    y_pred       = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

    mlflow.log_metrics({
        'test_accuracy':       accuracy_score(y_test_enc, y_pred),
        'test_auc':            roc_auc_score(y_test_enc, y_pred_proba),
        'test_precision_used': precision_score(y_test_enc, y_pred, pos_label=0),
        'test_recall_used':    recall_score(y_test_enc, y_pred, pos_label=0),
        'test_f1_used':        f1_score(y_test_enc, y_pred, pos_label=0),
        'test_precision_new':  precision_score(y_test_enc, y_pred, pos_label=1),
        'test_recall_new':     recall_score(y_test_enc, y_pred, pos_label=1),
    })

    mlflow.sklearn.log_model(pipeline, name='model')
    print(f"Run ID: {mlflow.active_run().info.run_id}")


# --- Métricas ---
print(classification_report(y_test_enc, y_pred, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_pred_proba):.4f}")


/Users/patriciogarbino/.pyenv/versions/meli-challenge/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/18 13:59:51 INFO mlflow.tracking.fluent: Experiment with name 'Baseline Model' does not exist. Creating a new experiment.
2026/03/18 13:59:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable

Run ID: 885620334c354d7c882e89262d912940
              precision    recall  f1-score   support

        used       0.80      0.81      0.80      4594
         new       0.83      0.83      0.83      5406

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000

AUC-ROC: 0.9011


#### Análisis de los resultados

El modelo alcanza una accuracy de **0.82**, quedando 4pp por debajo del mínimo requerido
de 0.86. Sin embargo, el AUC-ROC de **0.90** indica que el modelo tiene buena capacidad
discernir entre las dos clases

La **precisión de "used" es 0.80**, esta métrica es la que considero más importante para el modelo ya que creo que a nivel de negocio lo peor que puede hacer el algoritmo es decir que un producto usado es nuevo. 

El balance entre clases es adecuado — precisión y recall similares entre `new` y `used`
indican que el modelo no está sesgado hacia ninguna clase, lo cual es consistente con el
balance del dataset (~54% new, ~46% used).

### 3. Curva de aprendizaje — Regresión Logística

Se analiza cómo evoluciona el AUC del modelo a medida que aumenta el tamaño del conjunto
de entrenamiento, usando validación cruzada de 5 folds (`cv=5`). Quiero validar si serviría incorporar más datos al modelo para seguir mejorando

In [9]:
from sklearn.model_selection import learning_curve
from plotly import graph_objects as go

train_sizes, train_scores, val_scores = learning_curve(
    pipeline,  # tu pipeline de logistic regression
    X_train, y_train_enc,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=train_sizes, y=train_mean, name='Train AUC'))
fig.add_trace(go.Scatter(x=train_sizes, y=val_mean,   name='Val AUC'))
fig.update_layout(
    title='Curva de aprendizaje — Logistic Regression',
    xaxis_title='Tamaño del conjunto de entrenamiento',
    yaxis_title='AUC',
    template='plotly_white',
    height=500, width=900
)
fig.show()

#### Análisis de los resultados

Podemos ver que el modelo alcanzó con más de 70k registros casí el máximo de su performace, agregar más datos no mejoraría en gran medida el resultado.